## 1. Load Base Dataset

读取生成的基础数据，并检查数据大小和前几行。


In [32]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"D:\JupyterProject\AIML339_Toxicity")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chat_game_base.parquet"
)

df = pd.read_parquet(DATA_PATH)

print(df.shape)
display(df.head())

(1361583, 16)


,match_id,key,slot,time,unit,player_slot,snapshot_time,gold,xp,last_hits,message_order,prev_message_1,prev_message_2,prev_message_3,prev_message_4,prev_message_5
0,0,space created,1,5,Monkey,1,0,0,0,0,0,None,None,None,None,None
1,0,hah,1,6,Monkey,1,0,0,0,0,1,space created,None,None,None,None
2,0,ez 500,6,9,6k Slayer,129,0,0,0,0,2,hah,space created,None,None,None
3,0,mvp ulti,4,934,Kira,4,900,5910,5362,65,3,ez 500,hah,space created,None,None
4,0,bye,6,1486,6k Slayer,129,1440,5978,6705,69,4,mvp ulti,ez 500,hah,space created,None


## 2. Check Existing Features

检查基础数据中已经包含哪些字段，并确认之前生成的聊天历史和游戏状态特征已经成功保存。

In [33]:
print("Shape:", df.shape)

print("\nColumns:")
for col in df.columns:
    print(col)

existing_features = [
    "key",
    "prev_message_1",
    "prev_message_2",
    "prev_message_3",
    "prev_message_4",
    "prev_message_5",
    "gold",
    "xp",
    "last_hits"
]

available_features = [
    col for col in existing_features
    if col in df.columns
]

print("\nExisting context features:")
print(available_features)

display(df[available_features].head(10))

Shape: (1361583, 16)

Columns:
match_id
key
slot
time
unit
player_slot
snapshot_time
gold
xp
last_hits
message_order
prev_message_1
prev_message_2
prev_message_3
prev_message_4
prev_message_5

Existing context features:
['key', 'prev_message_1', 'prev_message_2', 'prev_message_3', 'prev_message_4', 'prev_message_5', 'gold', 'xp', 'last_hits']


,key,prev_message_1,prev_message_2,prev_message_3,prev_message_4,prev_message_5,gold,xp,last_hits
0,space created,None,None,None,None,None,0,0,0
1,hah,space created,None,None,None,None,0,0,0
2,ez 500,hah,space created,None,None,None,0,0,0
3,mvp ulti,ez 500,hah,space created,None,None,5910,5362,65
4,bye,mvp ulti,ez 500,hah,space created,None,5978,6705,69
5,hah,bye,mvp ulti,ez 500,hah,space created,8872,11034,77
6,fate,hah,bye,mvp ulti,ez 500,hah,5978,6705,69
7,is cruel,fate,hah,bye,mvp ulti,ez 500,6078,6861,69
8,fuck my ass,is cruel,fate,hah,bye,mvp ulti,6294,5841,15
9,ka bu tooooooooooooo,fuck my ass,is cruel,fate,hah,bye,7564,7747,20


## 3. Speaking Frequency

统计同一玩家在同一场比赛中，当前消息之前 60 秒内发送了多少条消息，用来表示短时间内的发言频率。

In [34]:
# Sort messages by match, player and time
df = df.sort_values(
    ["match_id", "player_slot", "time"]
).reset_index(drop=True)

# Convert seconds to timedelta for rolling window
df["time_td"] = pd.to_timedelta(
    df["time"],
    unit="s"
)

# Count messages sent by the same player in the previous 60 seconds
message_frequency = (
    df
    .groupby(
        ["match_id", "player_slot"],
        sort=False
    )
    .rolling(
        "60s",
        on="time_td",
        closed="left"
    )["key"]
    .count()
    .reset_index(level=[0, 1], drop=True)
    .fillna(0)
    .reset_index(drop=True)
)

df["messages_last_60s"] = (
    message_frequency
    .astype("int32")
)

display(
    df[
        [
            "match_id",
            "time",
            "player_slot",
            "key",
            "messages_last_60s"
        ]
    ].head(20)
)

,match_id,time,player_slot,key,messages_last_60s
0,0,1524,0,fuck my ass,0
1,0,1721,0,ka bu tooooooooooooo,0
2,0,2387,0,6k slayer,0
3,0,2388,0,haha,1
4,0,5,1,space created,0
5,0,6,1,hah,1
6,0,1488,1,hah,0
7,0,1854,1,wtf,0
8,0,1855,1,TA?,1
9,0,1857,1,u srsly?,2


In [35]:
print("Missing values:",
      df["messages_last_60s"].isna().sum())

print("Minimum:",
      df["messages_last_60s"].min())

print("Maximum:",
      df["messages_last_60s"].max())

Missing values: 0
Minimum: 0
Maximum: 102


## 4. Check Speaking Frequency

检查新生成的发言频率特征是否合理，并查看整体分布。

In [36]:
print(df["messages_last_60s"].describe())

print("\nHighest speaking frequency examples:")

display(
    df[
        [
            "match_id",
            "time",
            "player_slot",
            "key",
            "messages_last_60s"
        ]
    ]
    .sort_values(
        "messages_last_60s",
        ascending=False
    )
    .head(20)
)

count    1.361583e+06
mean     1.277891e+00
std      2.179208e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      1.020000e+02
Name: messages_last_60s, dtype: float64

Highest speaking frequency examples:


,match_id,time,player_slot,key,messages_last_60s
416147,14986,55,128,USA USA USA,102
416152,14986,55,128,USA USA USA,102
416153,14986,55,128,USA USA USA,102
416148,14986,55,128,USA USA USA,102
416149,14986,55,128,USA USA USA,102
416154,14986,55,128,ez mid,102
416145,14986,55,128,ez mid,102
416146,14986,55,128,USA USA USA,102
416151,14986,55,128,ez mid,102
416150,14986,55,128,USA USA USA,102


## 5. Clean Speaking Frequency

将过去 60 秒的发言次数转换为整数类型，方便后续模型使用。

In [37]:
df["messages_last_60s"] = (
    df["messages_last_60s"]
    .fillna(0)
    .astype("int32")
)

print(df["messages_last_60s"].dtype)

int32


## 6. Repeated Messages

检查同一玩家是否在 60 秒内重复发送相同内容，用来识别刷屏或重复表达行为。

In [38]:
df["normalized_message"] = (
    df["key"]
    .fillna("")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["previous_same_message_time"] = (
    df
    .groupby(
        ["match_id", "player_slot", "normalized_message"]
    )["time"]
    .shift(1)
)

df["repeat_time_gap"] = (
    df["time"] - df["previous_same_message_time"]
)

df["repeated_message_60s"] = (
    (df["repeat_time_gap"] >= 0) &
    (df["repeat_time_gap"] <= 60) &
    (df["normalized_message"] != "")
).astype("int8")

display(
    df[
        [
            "match_id",
            "time",
            "player_slot",
            "key",
            "messages_last_60s",
            "repeated_message_60s"
        ]
    ].head(20)
)

,match_id,time,player_slot,key,messages_last_60s,repeated_message_60s
0,0,1524,0,fuck my ass,0,0
1,0,1721,0,ka bu tooooooooooooo,0,0
2,0,2387,0,6k slayer,0,0
3,0,2388,0,haha,1,0
4,0,5,1,space created,0,0
5,0,6,1,hah,1,0
6,0,1488,1,hah,0,0
7,0,1854,1,wtf,0,0
8,0,1855,1,TA?,1,0
9,0,1857,1,u srsly?,2,0


## 7. Check Repeated Messages

检查重复消息特征的数量，并查看一些实际的重复发言案例。

In [39]:
print(
    df["repeated_message_60s"]
    .value_counts()
)

print("\nRepeated message examples:")

display(
    df[
        df["repeated_message_60s"] == 1
    ][
        [
            "match_id",
            "time",
            "player_slot",
            "key",
            "repeat_time_gap",
            "messages_last_60s"
        ]
    ].head(30)
)

repeated_message_60s
0    1331581
1      30002
Name: count, dtype: int64

Repeated message examples:


,match_id,time,player_slot,key,repeat_time_gap,messages_last_60s
15,0,2133,1,hah,21.0,1
135,6,2128,1,AS MIDLANER,40.0,6
311,12,2217,0,BOM,2.0,3
312,12,2218,0,BAm,3.0,4
323,12,3410,2,YES,1.0,2
343,13,2456,1,ez,14.0,2
344,13,2457,1,ez,1.0,3
345,13,2461,1,Ez,4.0,4
445,17,3818,132,yeah,13.0,1
678,28,1480,4,Ц,0.0,1


## 8. Message Interval

计算同一玩家两次发言之间的时间间隔，用来表示玩家是否正在短时间内连续发言。

In [40]:
df["previous_message_time"] = (
    df
    .groupby(
        ["match_id", "player_slot"]
    )["time"]
    .shift(1)
)

df["message_interval"] = (
    df["time"] - df["previous_message_time"]
)

display(
    df[
        [
            "match_id",
            "time",
            "player_slot",
            "key",
            "previous_message_time",
            "message_interval",
            "messages_last_60s"
        ]
    ].head(30)
)

,match_id,time,player_slot,key,previous_message_time,message_interval,messages_last_60s
0,0,1524,0,fuck my ass,NaN,NaN,0
1,0,1721,0,ka bu tooooooooooooo,1524.0,197.0,0
2,0,2387,0,6k slayer,1721.0,666.0,0
3,0,2388,0,haha,2387.0,1.0,1
4,0,5,1,space created,NaN,NaN,0
5,0,6,1,hah,5.0,1.0,1
6,0,1488,1,hah,6.0,1482.0,0
7,0,1854,1,wtf,1488.0,366.0,0
8,0,1855,1,TA?,1854.0,1.0,1
9,0,1857,1,u srsly?,1855.0,2.0,2


## 9. Check Message Interval

检查玩家发言间隔的整体分布，并查看短时间连续发言的实际案例。

In [41]:
print(df["message_interval"].describe())

print("\nRapid message examples:")

display(
    df[
        df["message_interval"].between(0, 5)
    ][
        [
            "match_id",
            "time",
            "player_slot",
            "key",
            "message_interval",
            "messages_last_60s",
            "repeated_message_60s"
        ]
    ].head(30)
)

count    1.064977e+06
mean     1.884839e+02
std      4.242040e+02
min      0.000000e+00
25%      3.000000e+00
50%      1.300000e+01
75%      1.400000e+02
max      9.218000e+03
Name: message_interval, dtype: float64

Rapid message examples:


,match_id,time,player_slot,key,message_interval,messages_last_60s,repeated_message_60s
3,0,2388,0,haha,1.0,1,0
5,0,6,1,hah,1.0,1,0
8,0,1855,1,TA?,1.0,1,0
9,0,1857,1,u srsly?,2.0,2,0
11,0,1909,1,what did i do,4.0,4,0
17,0,2143,1,alche,1.0,3,0
18,0,2145,1,hahahhaHAHA,2.0,4,0
28,0,1885,129,noes cape for him,3.0,1,0
34,0,2146,129,but we dive much any other,5.0,2,0
36,0,2312,129,u dont have a 6k mmr player with u,5.0,1,0


## 10. Inspect Match Events

读取团战和比赛目标事件数据，检查可以用于聊天上下文的时间和事件信息。

In [42]:
RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "dota2_matches"
)

teamfights = pd.read_csv(
    RAW_DATA_DIR / "teamfights.csv"
)

objectives = pd.read_csv(
    RAW_DATA_DIR / "objectives.csv"
)

print("Teamfights shape:")
print(teamfights.shape)

print("\nTeamfights columns:")
print(teamfights.columns.tolist())

print("\nTeamfights sample:")
display(teamfights.head(10))


print("\nObjectives shape:")
print(objectives.shape)

print("\nObjectives columns:")
print(objectives.columns.tolist())

print("\nObjectives sample:")
display(objectives.head(10))

Teamfights shape:
(539047, 5)

Teamfights columns:
['match_id', 'start', 'end', 'last_death', 'deaths']

Teamfights sample:


,match_id,start,end,last_death,deaths
0,0,220,252,237,3
1,0,429,475,460,3
2,0,900,936,921,3
3,0,1284,1328,1313,3
4,0,1614,1666,1651,5
5,0,1672,1709,1694,3
6,0,1734,1783,1768,6
7,0,1818,1867,1852,5
8,0,1863,1912,1897,5
9,0,2101,2145,2130,4



Objectives shape:
(1173396, 9)

Objectives columns:
['match_id', 'key', 'player1', 'player2', 'slot', 'subtype', 'team', 'time', 'value']

Objectives sample:


,match_id,key,player1,player2,slot,subtype,team,time,value
0,0,NaN,0,6,0.0,CHAT_MESSAGE_FIRSTBLOOD,NaN,1,309
1,0,NaN,3,-1,3.0,CHAT_MESSAGE_TOWER_KILL,2.0,894,2
2,0,NaN,2,-1,NaN,CHAT_MESSAGE_ROSHAN_KILL,2.0,925,200
3,0,NaN,1,-1,1.0,CHAT_MESSAGE_AEGIS,NaN,925,0
4,0,NaN,7,-1,7.0,CHAT_MESSAGE_TOWER_KILL,3.0,1016,3
5,0,NaN,3,-1,3.0,CHAT_MESSAGE_TOWER_KILL,2.0,1024,2
6,0,NaN,-1,-1,-1.0,CHAT_MESSAGE_TOWER_KILL,2.0,1446,2
7,0,NaN,4,-1,4.0,CHAT_MESSAGE_TOWER_KILL,2.0,1590,2
8,0,NaN,2,-1,NaN,CHAT_MESSAGE_ROSHAN_KILL,2.0,1740,200
9,0,NaN,1,-1,1.0,CHAT_MESSAGE_AEGIS,NaN,1740,0


## 11. Check Event Coverage

检查团战和比赛事件覆盖了多少场比赛，以及是否存在缺失值。

In [43]:
print("Matches with teamfight data:")
print(teamfights["match_id"].nunique())

print("\nMatches with objective data:")
print(objectives["match_id"].nunique())

print("\nTeamfight missing values:")
print(teamfights.isna().sum())

print("\nObjective missing values:")
print(objectives.isna().sum())

Matches with teamfight data:
49931

Matches with objective data:
49999

Teamfight missing values:
match_id      0
start         0
end           0
last_death    0
deaths        0
dtype: int64

Objective missing values:
match_id         0
key         903820
player1          0
player2          0
slot        346543
subtype          0
team        394641
time             0
value            0
dtype: int64


## 12. Inspect Objective Types

查看不同比赛事件的类型和数量，确定哪些事件适合作为聊天上下文特征。

In [44]:
objective_types = (
    objectives["subtype"]
    .value_counts()
    .reset_index()
)

objective_types.columns = [
    "subtype",
    "count"
]

display(objective_types)

,subtype,count
0,CHAT_MESSAGE_TOWER_KILL,663032
1,CHAT_MESSAGE_BARRACKS_KILL,269576
2,CHAT_MESSAGE_ROSHAN_KILL,76967
3,CHAT_MESSAGE_AEGIS,75174
4,CHAT_MESSAGE_FIRSTBLOOD,48721
5,CHAT_MESSAGE_TOWER_DENY,38756
6,CHAT_MESSAGE_AEGIS_STOLEN,1170


## 13. Inspect Teamfight Player Data

检查团战中的玩家数据，看看是否可以判断玩家所在队伍在最近团战中的表现。

In [45]:
teamfight_players = pd.read_csv(
    RAW_DATA_DIR / "teamfights_players.csv"
)

print("Shape:")
print(teamfight_players.shape)

print("\nColumns:")
print(teamfight_players.columns.tolist())

print("\nSample:")
display(teamfight_players.head(20))

Shape:
(5390470, 8)

Columns:
['match_id', 'player_slot', 'buybacks', 'damage', 'deaths', 'gold_delta', 'xp_end', 'xp_start']

Sample:


,match_id,player_slot,buybacks,damage,deaths,gold_delta,xp_end,xp_start
0,0,0,0,105,0,173,536,314
1,0,1,0,566,1,0,1583,1418
2,0,2,0,0,0,0,391,391
3,0,3,0,0,0,123,1775,1419
4,0,4,0,444,0,336,1267,983
5,0,128,0,477,1,249,1318,1035
6,0,129,0,636,1,-27,1048,904
7,0,130,0,0,0,190,1904,1589
8,0,131,0,0,0,0,210,210
9,0,132,0,0,0,378,659,589


## 14. Build Teamfight IDs

为每场团战和对应的玩家数据建立统一编号，方便计算双方在团战中的死亡情况。

In [46]:
teamfights = teamfights.copy()
teamfight_players = teamfight_players.copy()

# Number each teamfight inside the same match
teamfights["teamfight_id"] = (
    teamfights
    .groupby("match_id")
    .cumcount()
)

# Every teamfight has 10 player rows
teamfight_players["teamfight_id"] = (
    teamfight_players
    .groupby("match_id")
    .cumcount() // 10
)

print("Teamfights:")
display(
    teamfights[
        [
            "match_id",
            "teamfight_id",
            "start",
            "end",
            "deaths"
        ]
    ].head(10)
)

print("\nTeamfight players:")
display(
    teamfight_players[
        [
            "match_id",
            "teamfight_id",
            "player_slot",
            "deaths",
            "damage",
            "gold_delta"
        ]
    ].head(20)
)

Teamfights:


,match_id,teamfight_id,start,end,deaths
0,0,0,220,252,3
1,0,1,429,475,3
2,0,2,900,936,3
3,0,3,1284,1328,3
4,0,4,1614,1666,5
5,0,5,1672,1709,3
6,0,6,1734,1783,6
7,0,7,1818,1867,5
8,0,8,1863,1912,5
9,0,9,2101,2145,4



Teamfight players:


,match_id,teamfight_id,player_slot,deaths,damage,gold_delta
0,0,0,0,0,105,173
1,0,0,1,1,566,0
2,0,0,2,0,0,0
3,0,0,3,0,0,123
4,0,0,4,0,444,336
5,0,0,128,1,477,249
6,0,0,129,1,636,-27
7,0,0,130,0,0,190
8,0,0,131,0,0,0
9,0,0,132,0,0,378


## 15. Check Teamfight Mapping

检查每场团战是否正好对应 10 个玩家。

In [47]:
players_per_fight = (
    teamfight_players
    .groupby(["match_id", "teamfight_id"])
    .size()
)

print(players_per_fight.describe())

print(
    "\nAll teamfights have 10 players:",
    (players_per_fight == 10).all()
)

count    539047.0
mean         10.0
std           0.0
min          10.0
25%          10.0
50%          10.0
75%          10.0
max          10.0
dtype: float64

All teamfights have 10 players: True


## 16. Teamfight Result

统计每场团战双方的死亡人数，用来判断哪一方在团战中损失更大。

In [48]:
def get_team(player_slot):
    if 0 <= player_slot <= 4:
        return "radiant"
    elif 128 <= player_slot <= 132:
        return "dire"
    return None


teamfight_players["team"] = (
    teamfight_players["player_slot"]
    .apply(get_team)
)

teamfight_deaths = (
    teamfight_players
    .groupby(
        ["match_id", "teamfight_id", "team"]
    )["deaths"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

teamfight_summary = teamfights.merge(
    teamfight_deaths,
    on=["match_id", "teamfight_id"],
    how="left"
)

teamfight_summary["radiant_lost"] = (
    teamfight_summary["radiant"]
    > teamfight_summary["dire"]
)

teamfight_summary["dire_lost"] = (
    teamfight_summary["dire"]
    > teamfight_summary["radiant"]
)

display(
    teamfight_summary[
        [
            "match_id",
            "teamfight_id",
            "start",
            "end",
            "radiant",
            "dire",
            "radiant_lost",
            "dire_lost"
        ]
    ].head(20)
)

,match_id,teamfight_id,start,end,radiant,dire,radiant_lost,dire_lost
0,0,0,220,252,1,2,False,True
1,0,1,429,475,0,3,False,True
2,0,2,900,936,0,3,False,True
3,0,3,1284,1328,0,3,False,True
4,0,4,1614,1666,2,3,False,True
5,0,5,1672,1709,3,0,True,False
6,0,6,1734,1783,2,4,False,True
7,0,7,1818,1867,2,3,False,True
8,0,8,1863,1912,2,3,False,True
9,0,9,2101,2145,1,3,False,True


## 17. Validate Teamfight Deaths

检查玩家团战数据中的死亡总数是否与团战表记录一致，确认团战编号映射正确。

In [49]:
player_death_totals = (
    teamfight_players
    .groupby(
        ["match_id", "teamfight_id"]
    )["deaths"]
    .sum()
    .reset_index(name="player_deaths_total")
)

death_check = teamfights[
    [
        "match_id",
        "teamfight_id",
        "deaths"
    ]
].merge(
    player_death_totals,
    on=["match_id", "teamfight_id"],
    how="left"
)

death_check["matched"] = (
    death_check["deaths"]
    == death_check["player_deaths_total"]
)

print(
    "Matched teamfights:",
    death_check["matched"].sum()
)

print(
    "Total teamfights:",
    len(death_check)
)

print(
    "Match rate:",
    death_check["matched"].mean()
)

display(
    death_check[
        death_check["matched"] == False
    ].head(20)
)

Matched teamfights: 513559
Total teamfights: 539047
Match rate: 0.9527165534730738


,match_id,teamfight_id,deaths,player_deaths_total,matched
49,3,11,3,4,False
62,5,2,3,4,False
122,10,4,4,5,False
147,12,11,4,5,False
148,12,12,3,4,False
153,12,17,3,4,False
158,13,4,3,4,False
204,17,8,5,7,False
215,18,5,3,4,False
240,22,4,4,5,False


## 17.1 Check Death Count Differences

检查团战总表和玩家表中的死亡统计差异。两张表可能使用不同的死亡统计方式，因此玩家级数据将用于计算双方团战损失。

In [50]:
death_check["death_difference"] = (
    death_check["player_deaths_total"]
    - death_check["deaths"]
)

print(
    death_check["death_difference"]
    .value_counts()
    .sort_index()
)

death_difference
-1         1
 0    513559
 1     24241
 2      1122
 3        56
 4        20
 5        44
 6         4
Name: count, dtype: int64


## 18. Add Player Team

根据 player slot 判断聊天玩家属于 Radiant 还是 Dire，为后续判断最近团战是否失利做准备。

In [51]:
def get_player_team(player_slot):
    if 0 <= player_slot <= 4:
        return "radiant"
    elif 128 <= player_slot <= 132:
        return "dire"
    return None


df["player_team"] = (
    df["player_slot"]
    .apply(get_player_team)
)

print(df["player_team"].value_counts())

player_team
dire       685416
radiant    676167
Name: count, dtype: int64


## 19. Prepare Teamfight Context

整理每场团战的结束时间和结果，用于寻找每条聊天之前最近发生的团战。

In [52]:
teamfight_events = teamfight_summary[
    [
        "match_id",
        "teamfight_id",
        "start",
        "end",
        "radiant",
        "dire",
        "radiant_lost",
        "dire_lost"
    ]
].copy()

teamfight_events = teamfight_events.rename(
    columns={
        "start": "teamfight_start",
        "end": "teamfight_end",
        "radiant": "radiant_deaths",
        "dire": "dire_deaths"
    }
)

display(teamfight_events.head(20))

,match_id,teamfight_id,teamfight_start,teamfight_end,radiant_deaths,dire_deaths,radiant_lost,dire_lost
0,0,0,220,252,1,2,False,True
1,0,1,429,475,0,3,False,True
2,0,2,900,936,0,3,False,True
3,0,3,1284,1328,0,3,False,True
4,0,4,1614,1666,2,3,False,True
5,0,5,1672,1709,3,0,True,False
6,0,6,1734,1783,2,4,False,True
7,0,7,1818,1867,2,3,False,True
8,0,8,1863,1912,2,3,False,True
9,0,9,2101,2145,1,3,False,True


## 20. Match Recent Teamfight to Chat

为每条聊天找到同一场比赛中，在发言之前最近结束的一场团战。

In [53]:
# Keep the original row order
chat_for_merge = df.copy()

chat_for_merge["original_order"] = range(
    len(chat_for_merge)
)

# merge_asof requires the time key to be globally sorted
chat_sorted = (
    chat_for_merge
    .sort_values(
        ["time", "match_id"]
    )
    .reset_index(drop=True)
)

teamfight_sorted = (
    teamfight_events
    .sort_values(
        ["teamfight_end", "match_id"]
    )
    .reset_index(drop=True)
)

df_with_teamfight = pd.merge_asof(
    chat_sorted,
    teamfight_sorted,
    left_on="time",
    right_on="teamfight_end",
    by="match_id",
    direction="backward"
)

# Restore the original chat order
df_with_teamfight = (
    df_with_teamfight
    .sort_values("original_order")
    .drop(columns="original_order")
    .reset_index(drop=True)
)

print("Rows before:", len(df))
print("Rows after:", len(df_with_teamfight))

display(
    df_with_teamfight[
        [
            "match_id",
            "time",
            "key",
            "player_team",
            "teamfight_end",
            "radiant_deaths",
            "dire_deaths"
        ]
    ].head(30)
)

Rows before: 1361583
Rows after: 1361583


,match_id,time,key,player_team,teamfight_end,radiant_deaths,dire_deaths
0,0,1524,fuck my ass,radiant,1328.0,0.0,3.0
1,0,1721,ka bu tooooooooooooo,radiant,1709.0,3.0,0.0
2,0,2387,6k slayer,radiant,2284.0,1.0,3.0
3,0,2388,haha,radiant,2388.0,1.0,2.0
4,0,5,space created,radiant,NaN,NaN,NaN
5,0,6,hah,radiant,NaN,NaN,NaN
6,0,1488,hah,radiant,1328.0,0.0,3.0
7,0,1854,wtf,radiant,1783.0,2.0,4.0
8,0,1855,TA?,radiant,1783.0,2.0,4.0
9,0,1857,u srsly?,radiant,1783.0,2.0,4.0


## 21. Create Teamfight Features

计算距离上一场团战结束的时间，并判断最近两分钟内是否发生团战以及玩家所在队伍是否在该团战中损失更大。

In [54]:
df_with_teamfight["seconds_since_teamfight"] = (
    df_with_teamfight["time"]
    - df_with_teamfight["teamfight_end"]
)

# Whether a teamfight ended within the previous 120 seconds
df_with_teamfight["recent_teamfight_120s"] = (
    df_with_teamfight["seconds_since_teamfight"]
    .between(0, 120)
    .astype("int8")
)

# Convert fight result columns to nullable boolean first
radiant_lost = (
    df_with_teamfight["radiant_lost"]
    .astype("boolean")
    .fillna(False)
)

dire_lost = (
    df_with_teamfight["dire_lost"]
    .astype("boolean")
    .fillna(False)
)

# Whether the player's team lost the most recent fight
df_with_teamfight["recent_teamfight_loss"] = (
    (
        (df_with_teamfight["player_team"] == "radiant")
        & radiant_lost
    )
    |
    (
        (df_with_teamfight["player_team"] == "dire")
        & dire_lost
    )
)

# Only count the loss if the fight ended within 120 seconds
df_with_teamfight["recent_teamfight_loss_120s"] = (
    df_with_teamfight["recent_teamfight_loss"]
    &
    (df_with_teamfight["recent_teamfight_120s"] == 1)
).astype("int8")

display(
    df_with_teamfight[
        [
            "match_id",
            "time",
            "key",
            "player_team",
            "teamfight_end",
            "seconds_since_teamfight",
            "recent_teamfight_120s",
            "recent_teamfight_loss_120s"
        ]
    ].head(30)
)

,match_id,time,key,player_team,teamfight_end,seconds_since_teamfight,recent_teamfight_120s,recent_teamfight_loss_120s
0,0,1524,fuck my ass,radiant,1328.0,196.0,0,0
1,0,1721,ka bu tooooooooooooo,radiant,1709.0,12.0,1,1
2,0,2387,6k slayer,radiant,2284.0,103.0,1,0
3,0,2388,haha,radiant,2388.0,0.0,1,0
4,0,5,space created,radiant,NaN,NaN,0,0
5,0,6,hah,radiant,NaN,NaN,0,0
6,0,1488,hah,radiant,1328.0,160.0,0,0
7,0,1854,wtf,radiant,1783.0,71.0,1,0
8,0,1855,TA?,radiant,1783.0,72.0,1,0
9,0,1857,u srsly?,radiant,1783.0,74.0,1,0


## 22. Check Teamfight Features

检查团战上下文特征的数量和时间关系，确保没有使用聊天之后发生的团战信息。

In [55]:
print("Recent teamfight within 120s:")
print(
    df_with_teamfight["recent_teamfight_120s"]
    .value_counts()
)

print("\nRecent teamfight loss within 120s:")
print(
    df_with_teamfight["recent_teamfight_loss_120s"]
    .value_counts()
)

valid_fights = df_with_teamfight["teamfight_end"].notna()

print(
    "\nAll matched teamfights are before chat:",
    (
        df_with_teamfight.loc[valid_fights, "teamfight_end"]
        <=
        df_with_teamfight.loc[valid_fights, "time"]
    ).all()
)

print(
    "\nMinimum seconds since teamfight:",
    df_with_teamfight["seconds_since_teamfight"].min()
)

Recent teamfight within 120s:
recent_teamfight_120s
1    793759
0    567824
Name: count, dtype: int64

Recent teamfight loss within 120s:
recent_teamfight_loss_120s
0    1005470
1     356113
Name: count, dtype: int64

All matched teamfights are before chat: True

Minimum seconds since teamfight: 0.0


## 23. Final Feature Check

检查最终上下文特征的数据类型、缺失值和基本分布，确认数据可以用于下一阶段。

In [56]:
final_features = [
    "key",
    "prev_message_1",
    "prev_message_2",
    "prev_message_3",
    "prev_message_4",
    "prev_message_5",
    "gold",
    "xp",
    "last_hits",
    "messages_last_60s",
    "repeated_message_60s",
    "message_interval",
    "seconds_since_teamfight",
    "recent_teamfight_120s",
    "recent_teamfight_loss_120s"
]

print("Dataset shape:")
print(df_with_teamfight.shape)

print("\nFeature data types:")
print(df_with_teamfight[final_features].dtypes)

print("\nMissing values:")
print(df_with_teamfight[final_features].isna().sum())

Dataset shape:
(1361583, 36)

Feature data types:
key                            object
prev_message_1                 object
prev_message_2                 object
prev_message_3                 object
prev_message_4                 object
prev_message_5                 object
gold                            int64
xp                              int64
last_hits                       int64
messages_last_60s               int32
repeated_message_60s             int8
message_interval              float64
seconds_since_teamfight       float64
recent_teamfight_120s            int8
recent_teamfight_loss_120s       int8
dtype: object

Missing values:
key                               13
prev_message_1                 49746
prev_message_2                 99037
prev_message_3                147640
prev_message_4                195348
prev_message_5                241973
gold                               0
xp                                 0
last_hits                          0
messages_last_60

## 24. Keep Final Context Features

保留后续标注和建模需要的字段，并删除只用于中间计算的临时变量。

In [57]:
columns_to_keep = [
    "match_id",
    "time",
    "player_slot",
    "unit",
    "key",

    "prev_message_1",
    "prev_message_2",
    "prev_message_3",
    "prev_message_4",
    "prev_message_5",

    "gold",
    "xp",
    "last_hits",

    "messages_last_60s",
    "repeated_message_60s",
    "message_interval",

    "player_team",
    "seconds_since_teamfight",
    "recent_teamfight_120s",
    "recent_teamfight_loss_120s"
]

final_context = (
    df_with_teamfight[
        columns_to_keep
    ]
    .copy()
)

print("Final shape:")
print(final_context.shape)

display(final_context.head(20))

Final shape:
(1361583, 20)


,match_id,time,player_slot,unit,key,prev_message_1,prev_message_2,prev_message_3,prev_message_4,prev_message_5,gold,xp,last_hits,messages_last_60s,repeated_message_60s,message_interval,player_team,seconds_since_teamfight,recent_teamfight_120s,recent_teamfight_loss_120s
0,0,1524,0,Double T,fuck my ass,is cruel,fate,hah,bye,mvp ulti,6294,5841,15,0,0,NaN,radiant,196.0,0,0
1,0,1721,0,Double T,ka bu tooooooooooooo,fuck my ass,is cruel,fate,hah,bye,7564,7747,20,0,0,197.0,radiant,12.0,1,1
2,0,2387,0,Double T,6k slayer,gG,wahaha,really ?,torso highest here,wlwlwlwllww,13409,13356,30,0,0,666.0,radiant,103.0,1,0
3,0,2388,0,Double T,haha,6k slayer,gG,wahaha,really ?,torso highest here,13409,13356,30,1,0,1.0,radiant,0.0,1,0
4,0,5,1,Monkey,space created,None,None,None,None,None,0,0,0,0,0,NaN,radiant,NaN,0,0
5,0,6,1,Monkey,hah,space created,None,None,None,None,0,0,0,1,0,1.0,radiant,NaN,0,0
6,0,1488,1,Monkey,hah,bye,mvp ulti,ez 500,hah,space created,8872,11034,77,0,0,1482.0,radiant,160.0,0,0
7,0,1854,1,Monkey,wtf,ka bu tooooooooooooo,fuck my ass,is cruel,fate,hah,12590,17244,83,0,0,366.0,radiant,71.0,1,0
8,0,1855,1,Monkey,TA?,wtf,ka bu tooooooooooooo,fuck my ass,is cruel,fate,12590,17244,83,1,0,1.0,radiant,72.0,1,0
9,0,1857,1,Monkey,u srsly?,TA?,wtf,ka bu tooooooooooooo,fuck my ass,is cruel,12590,17244,83,2,0,2.0,radiant,74.0,1,0


## 25. Save Context Dataset

保存完成的上下文数据，供下一阶段的三分类标注和模型训练使用。

In [58]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    PROCESSED_DIR
    / "chat_with_context.parquet"
)

final_context.to_parquet(
    output_path,
    index=False
)

print("Saved to:")
print(output_path)

print("\nSaved rows:")
print(len(final_context))

Saved to:
D:\JupyterProject\AIML339_Toxicity\data\processed\chat_with_context.parquet

Saved rows:
1361583


## 26. Verify Saved Dataset

重新读取保存的数据，确认文件可以正常使用。

In [59]:
check_df = pd.read_parquet(
    output_path
)

print("Loaded shape:")
print(check_df.shape)

print("\nSame number of rows:")
print(
    len(check_df)
    ==
    len(final_context)
)

display(check_df.head())

Loaded shape:
(1361583, 20)

Same number of rows:
True


,match_id,time,player_slot,unit,key,prev_message_1,prev_message_2,prev_message_3,prev_message_4,prev_message_5,gold,xp,last_hits,messages_last_60s,repeated_message_60s,message_interval,player_team,seconds_since_teamfight,recent_teamfight_120s,recent_teamfight_loss_120s
0,0,1524,0,Double T,fuck my ass,is cruel,fate,hah,bye,mvp ulti,6294,5841,15,0,0,NaN,radiant,196.0,0,0
1,0,1721,0,Double T,ka bu tooooooooooooo,fuck my ass,is cruel,fate,hah,bye,7564,7747,20,0,0,197.0,radiant,12.0,1,1
2,0,2387,0,Double T,6k slayer,gG,wahaha,really ?,torso highest here,wlwlwlwllww,13409,13356,30,0,0,666.0,radiant,103.0,1,0
3,0,2388,0,Double T,haha,6k slayer,gG,wahaha,really ?,torso highest here,13409,13356,30,1,0,1.0,radiant,0.0,1,0
4,0,5,1,Monkey,space created,None,None,None,None,None,0,0,0,0,0,NaN,radiant,NaN,0,0
